# ДЗ 1. Квартиры: разведочный анализ и модель цены

Условие — в репозитории курса: `assignments/hw01/README.md`.

Данные: `data/train.csv` (с ценой) и `data/test.csv` (без цены). Папка `data/` в git не попадает —
получите файлы в канале курса и положите рядом с ноутбуком.

Ноутбук должен выполняться **с нуля** командой «Restart & Run All» и создавать:
- `outputs/answers.json` — ответы на вопросы части 1;
- `outputs/predictions.csv` — предсказания для `test.csv` (колонки `id`, `price`).


In [ ]:
import json
import os

import numpy as np
import pandas as pd

train = pd.read_csv("data/train.csv")
test = pd.read_csv("data/test.csv")
os.makedirs("outputs", exist_ok=True)
train.head()

## Часть 1. Вопросы по данным (3 балла)

Заполните словарь `answers`. Числа округляйте как указано; названия районов — как в данных.

| ключ | что посчитать |
|---|---|
| `n_rows` | число строк в `train` |
| `median_price` | медиана `price`, до 1 знака |
| `top_district` | район с наибольшей **медианной** ценой |
| `corr_area_price` | корреляция Пирсона между `area` и `price`, до 3 знаков |
| `share_first_floor` | доля квартир на 1-м этаже, до 3 знаков |
| `mean_price_per_m2` | среднее значение `price / area`, до 1 знака |


In [ ]:
answers = {
    "n_rows": int(len(train)),
    "median_price": float(round(train["price"].median(), 1)),
    "top_district": str(train.groupby("district")["price"].median().idxmax()),
    "corr_area_price": float(round(train["area"].corr(train["price"]), 3)),
    "share_first_floor": float(round((train["floor"] == 1).mean(), 3)),
    "mean_price_per_m2": float(round((train["price"] / train["area"]).mean(), 1)),
}

with open("outputs/answers.json", "w", encoding="utf-8") as f:
    json.dump(answers, f, ensure_ascii=False, indent=2)
answers

## Часть 2. Модель цены (3 балла + 1 за формат)

Обучите модель на `train` и предскажите `price` для всех объектов `test`. Метрика — $R^2$ на скрытых
ответах: $R^2 \ge 0.90$ — 3 балла, $\ge 0.80$ — 2, $\ge 0.60$ — 1.

Файл `outputs/predictions.csv` должен содержать ровно те `id`, что в `test.csv`, без пропусков и дублей.
Для честной оценки качества сделайте у себя отложенную выборку или кросс-валидацию.


In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.metrics import r2_score
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder

features = ["district", "area", "rooms", "floor", "total_floors", "year_built", "metro_min", "has_balcony"]
X, y = train[features], train["price"]

model = Pipeline([
    ("prep", ColumnTransformer([("district", OneHotEncoder(handle_unknown="ignore"), ["district"])],
                               remainder="passthrough")),
    ("gbr", GradientBoostingRegressor(n_estimators=400, max_depth=3, learning_rate=0.05, random_state=0)),
])

X_tr, X_val, y_tr, y_val = train_test_split(X, y, test_size=0.2, random_state=0)
model.fit(X_tr, y_tr)
print("R2 на отложенной выборке:", round(r2_score(y_val, model.predict(X_val)), 4))

model.fit(X, y)
predictions = pd.DataFrame({"id": test["id"], "price": model.predict(test[features])})
predictions.to_csv("outputs/predictions.csv", index=False)
predictions.head()

## Часть 3. Отчёт (1 балл)

Напишите в `report.md` (в корне репозитория) короткие выводы: какие признаки сильнее всего влияют на цену,
насколько хороша модель и где она ошибается. Не меньше 300 символов. Отчёт читает проверяющий.
